## Setup
First, let's install the required packages and set up the API keys

In [26]:
%%capture --no-stderr
%pip install -U langchain-openai langgraph langgraph-checkpoint-redis

In [27]:
import getpass
import os

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("OPENAI_API_KEY")

## Graph Implementation
Let's implement the graph that will leverage Redis for storing state transitions.

In [28]:
from typing import Literal
from IPython.display import Image, display

from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.redis import RedisSaver

# Define a simple tool
@tool
def get_weather(city: Literal["nyc", "sf"]):
    """Use this to get weather information."""
    if city == "nyc":
        return "It might be cloudy in nyc"
    elif city == "sf":
        return "It's always sunny in sf"
    else:
        raise AssertionError("Unknown city")


In [29]:
# Set up model and tools
tools = [get_weather]
model = ChatOpenAI(model = "gpt-4o-mini", temperature = 0)

In [30]:
from typing import Literal

from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.redis import RedisSaver

# Define a simple tool
@tool
def get_weather(city: Literal["nyc", "sf"]):
    """Use this to get weather information."""
    if city == "nyc":
        return "It might be cloudy in nyc"
    elif city == "sf":
        return "It's always sunny in sf"
    else:
        raise AssertionError("Unknown city")

# Set up model and tools
tools = [get_weather]
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Create Redis persistence
REDIS_URI = "redis://localhost:6379"
with RedisSaver.from_conn_string(REDIS_URI) as checkpointer:
    # Initialize Redis indices (only needed once)
    checkpointer.setup()
    
    # Create agent with memory
    graph = create_react_agent(model, tools=tools, checkpointer=checkpointer)
    
    
    # Use the agent with a specific thread ID to maintain conversation state
    config = {"configurable": {"thread_id": "user2322"}}
    res = graph.invoke({"messages": [("human", "Which state did I ask?")]}, config)
    
    # Extract clean response - get the last AI message content
    res = res["messages"][-1].content


13:32:16 langgraph.checkpoint.redis INFO   Redis client is a standalone client
13:32:16 redisvl.index.index INFO   Index already exists, not overwriting.
13:32:16 redisvl.index.index INFO   Index already exists, not overwriting.
13:32:16 redisvl.index.index INFO   Index already exists, not overwriting.


/var/folders/_9/wkm0tqns0n53w3ytpgc_68kc0000gn/T/ipykernel_37987/1173132020.py:30: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  graph = create_react_agent(model, tools=tools, checkpointer=checkpointer)


13:32:17 httpx INFO   HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


In [31]:
print(res)

You asked about the weather in San Francisco (SF), which is located in California.
